In [38]:
%load_ext autoreload
%autoreload 2

from radar.components import geometry
from radar.utils.calculate import convert
from radar.utils.typing.enums import FrequencyUnit
from radar.utils.typing.units import Frequency

from radar.components import Element
from radar.utils.calculate import pattern
from radar.utils.typing import (
    PhaseUnit,
    DirectionDomain,
    FigureType,
    AmplitudeDomain,
    Angle,
    AmplitudeUnit,
)

from radar.components.array import Array
import polars as pl

from radar.utils.typing.constants import DataHeader

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
beam_width = 30
beam_width_tuple = (
    Angle(beam_width, PhaseUnit.DEGREE),
    Angle(beam_width, PhaseUnit.DEGREE),
)

az_bound = 90
el_bound = 90
az_bound_tuple = (Angle(-az_bound, PhaseUnit.DEGREE), Angle(az_bound, PhaseUnit.DEGREE))
el_bound_tuple = (Angle(-el_bound, PhaseUnit.DEGREE), Angle(el_bound, PhaseUnit.DEGREE))


element_pattern = pattern.Isotropic()
freq = Frequency(1, FrequencyUnit.GIGAHERTZ)
antenna_element = Element(element_pattern, az_bound_tuple, el_bound_tuple, freq, 1)

cf = Frequency(1, FrequencyUnit.GIGAHERTZ)
distance = convert.cf_to_min_dist(cf)
array_geometry = geometry.Grid(5, 5, distance)

arr = Array(antenna_element, array_geometry)

arr.plot.beam(
    DirectionDomain.ANGLE,
    PhaseUnit.DEGREE,
    AmplitudeDomain.AntennaFactor,
    AmplitudeUnit.DECIBEL,
    FigureType.SURFACE,
    Frequency(1, FrequencyUnit.GIGAHERTZ),
)

In [40]:
# import array

# from radar.optimiser.biology import Organism

# def calculate_genetic_health(org: Organism) -> float:
#     score = 0.0
    
#     for chromosome in org.chromosomes:
#         # Access the underlying Polars DataFrame
#         # df = chromosome.df  

#         az_bound = 90
#         el_bound = 90
#         az_bound_tuple = (Angle(-az_bound, PhaseUnit.DEGREE), Angle(az_bound, PhaseUnit.DEGREE))
#         el_bound_tuple = (Angle(-el_bound, PhaseUnit.DEGREE), Angle(el_bound, PhaseUnit.DEGREE))


#         element_pattern = pattern.Isotropic()
#         freq = Frequency(1, FrequencyUnit.GIGAHERTZ)
#         antenna_element = Element(element_pattern, az_bound_tuple, el_bound_tuple, freq, 1)

#         cf = Frequency(1, FrequencyUnit.GIGAHERTZ)
#         distance = convert.cf_to_min_dist(cf)
#         array_geometry = geometry.Grid(5, 5, distance)
#         # 1. Start with the original DataFrame
#         df = array_geometry.df

#         # print(sum(df[DataHeader.GEOM_AMP_GAIN_DB]))
#         # 2. Chain the operations together so the data flows seamlessly
#         updated_df = (
#             df.with_columns(
#                 chromosome.df.to_series().alias(DataHeader.GEOM_AMP_GAIN_DB)
#             )
#             .drop(DataHeader.GEOM_AMP_GAIN_LIN)
#         )

#         array_geometry.gains = updated_df

#         arr = Array(antenna_element, array_geometry)
        
#         return arr.statistic.std(cf)

        
#     return score

In [ ]:
import array
from typing import cast
import polars as pl

from radar.optimiser.biology import Organism
from IPython.display import clear_output

def calculate_genetic_health2(org: Organism, generation : int | None = None) -> float:

    x = org._chromosomes[0]
    y = org._chromosomes[1]

    az_bound = 90
    el_bound = 90
    az_bound_tuple = (Angle(-az_bound, PhaseUnit.DEGREE), Angle(az_bound, PhaseUnit.DEGREE))
    el_bound_tuple = (Angle(-el_bound, PhaseUnit.DEGREE), Angle(el_bound, PhaseUnit.DEGREE))


    element_pattern = pattern.Isotropic()
    freq = Frequency(1, FrequencyUnit.GIGAHERTZ)
    antenna_element = Element(element_pattern, az_bound_tuple, el_bound_tuple, freq, 1)

    cf = Frequency(1, FrequencyUnit.GIGAHERTZ)
    array_geometry = geometry.CustomGeometry(x.df.to_numpy().ravel(), y.df.to_numpy().ravel())

    arr = Array(antenna_element, array_geometry)
    
    beam = arr.beam_pattern(cf, None)
    # Fixed: Added parentheses around conditions and used native Polars .abs()
    ave = (
        beam.filter(
            (pl.col(DataHeader.AZIMUTH_DEG).abs() > 20) & 
            (pl.col(DataHeader.ELEVATION_DEG).abs() > 20)
        )
        .select(DataHeader.ANTENNA_FACTOR_DB)
        .to_series()
        .mean()
    )

    ave_beam = (
        beam.filter(
            (pl.col(DataHeader.AZIMUTH_DEG).abs() < 20) & 
            (pl.col(DataHeader.ELEVATION_DEG).abs() < 20)
        )
        .select(DataHeader.ANTENNA_FACTOR_DB)
        .to_series()
        .mean()
    )  
    ave = cast(float, ave)
    ave_beam = cast(float, ave_beam)
    std = arr.statistic.std(cf) 

    if generation and generation % 5 == 0:
        clear_output(wait=True) 
        arr.plot.beam(
            DirectionDomain.ANGLE,
            PhaseUnit.DEGREE,
            AmplitudeDomain.AntennaFactor,
            AmplitudeUnit.DECIBEL,
            FigureType.SURFACE,
            Frequency(1, FrequencyUnit.GIGAHERTZ),
            None,
            f"tmp/beam_{generation}"
        )

        print(f"-----  {ave} {ave_beam} {std}")


    
    # lower is better

   
    return std + 4*(ave - ave_beam) # arr.statistic.ave(cf)


In [51]:
tmp_org = best_organism

In [54]:
#1. Initialize your population
from radar.optimiser.biology import Population

pop = Population(
    pop_size=200, 
    headers=[DataHeader.X_POS_M, DataHeader.Y_POS_M], 
    num_values_per_chrom=49, 
    min_val=-0.5, 
    max_val=0.5,
    kill_prop=0.9
)

In [55]:
pop.organisms[0] = tmp_org

In [ ]:
import datetime

import logging

# Mute the kaleido logger specifically
logging.getLogger("kaleido").setLevel(logging.ERROR)
# # 2. Define how many generations you want to evolve
num_generations = 5000

print("Starting optimization loop...\n")

for generation in range(1, num_generations + 1):
    # 3. Propagate the population to the next generation
    # Pass the function name without parentheses!
    pop.propagate(fitness_fn=calculate_genetic_health2)
    
    # 4. Optional: Inspect the best-performing organism of this generation
    best_organism = pop.organisms[0]
    
    # Let's say your fitness function is accessible or you want to see its score
    best_score = calculate_genetic_health2(best_organism, generation)

    print(f"Generation {generation}/{num_generations}: Best Fitness Score = {best_score:.8f}")

print("\nOptimization complete!")
# pop.organisms[0] now contains your ultimate optimized organism

[06/15/26 13:12:37] INFO     Chromium init'ed with kwargs {}                                        ]8;id=10150381;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/browsers/chromium.py\chromium.py]8;;\:]8;id=10150382;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/browsers/chromium.py#156\156]8;;\

                    INFO     Found chromium path: /usr/bin/chromium                                 ]8;id=10150387;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/browsers/chromium.py\chromium.py]8;;\:]8;id=10150388;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/browsers/chromium.py#184\184]8;;\

                    INFO     Temp directory created: /tmp/tmpjr14nvle.                               ]8;id=10150393;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/utils/_tmpfile.py\_tmpfile.py]8;;\:]8;id=10150394;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/utils/_tmpfile.py#80\80]8;;\

                    INFO     Opening browser.                                                  ]8;id=10150399;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/browser_async.py\browser_async.py]8;;\:]8;id=10150400;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/browser_async.py#127\127]8;;\

                    INFO     Temp directory created: /tmp/tmpnpoym8xp.                               ]8;id=10150405;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/utils/_tmpfile.py\_tmpfile.py]8;;\:]8;id=10150406;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/utils/_tmpfile.py#80\80]8;;\

...

                    INFO     TemporaryDirectory.cleanup() worked.                                   ]8;id=10150453;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/utils/_tmpfile.py\_tmpfile.py]8;;\:]8;id=10150454;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/utils/_tmpfile.py#156\156]8;;\

                    INFO     shutil.rmtree worked.                                                  ]8;id=10150459;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/utils/_tmpfile.py\_tmpfile.py]8;;\:]8;id=10150460;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/utils/_tmpfile.py#180\180]8;;\

-----  -9.683640405719478 -6.971996686210335 4.656761740698529
Generation 140/5000: Best Fitness Score = -6.18981314


[06/15/26 13:12:44] INFO     TemporaryDirectory.cleanup() worked.                                   ]8;id=10150465;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/utils/_tmpfile.py\_tmpfile.py]8;;\:]8;id=10150466;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/utils/_tmpfile.py#156\156]8;;\

                    INFO     shutil.rmtree worked.                                                  ]8;id=10150471;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/utils/_tmpfile.py\_tmpfile.py]8;;\:]8;id=10150472;file:///home/gnorrie/Work/radar-sdk/.venv/lib/python3.14/site-packages/choreographer/utils/_tmpfile.py#180\180]8;;\

In [47]:
# from radar.optimiser.biology import chromosome


chromosome = best_organism.chromosomes[0]
az_bound = 90
el_bound = 90
az_bound_tuple = (Angle(-az_bound, PhaseUnit.DEGREE), Angle(az_bound, PhaseUnit.DEGREE))
el_bound_tuple = (Angle(-el_bound, PhaseUnit.DEGREE), Angle(el_bound, PhaseUnit.DEGREE))


element_pattern = pattern.Isotropic()
freq = Frequency(1, FrequencyUnit.GIGAHERTZ)
antenna_element = Element(element_pattern, az_bound_tuple, el_bound_tuple, freq, 1)

cf = Frequency(1, FrequencyUnit.GIGAHERTZ)
distance = convert.cf_to_min_dist(cf)
array_geometry = geometry.Grid(5, 5, distance)
# 1. Start with the original DataFrame
df = array_geometry.df

# print(sum(df[DataHeader.GEOM_AMP_GAIN_DB]))
# 2. Chain the operations together so the data flows seamlessly
updated_df = (
    df.with_columns(
        chromosome.df.to_series().alias(DataHeader.GEOM_AMP_GAIN_DB)
    )
    .drop(DataHeader.GEOM_AMP_GAIN_LIN)
)

array_geometry.gains = updated_df

arr = Array(antenna_element, array_geometry)

arr.plot.beam(
    DirectionDomain.ANGLE,
    PhaseUnit.DEGREE,
    AmplitudeDomain.AntennaFactor,
    AmplitudeUnit.DECIBEL,
    FigureType.SURFACE,
    Frequency(1, FrequencyUnit.GIGAHERTZ),
)

arr.plot.geometry()

arr._geometry.df[DataHeader.GEOM_AMP_GAIN_DB]

ShapeError: unable to add a column of length 49 to a DataFrame of height 25

In [48]:
x = best_organism._chromosomes[0]
y = best_organism._chromosomes[1]

az_bound = 90
el_bound = 90
az_bound_tuple = (Angle(-az_bound, PhaseUnit.DEGREE), Angle(az_bound, PhaseUnit.DEGREE))
el_bound_tuple = (Angle(-el_bound, PhaseUnit.DEGREE), Angle(el_bound, PhaseUnit.DEGREE))


element_pattern = pattern.Isotropic()
freq = Frequency(1, FrequencyUnit.GIGAHERTZ)
antenna_element = Element(element_pattern, az_bound_tuple, el_bound_tuple, freq, 1)

cf = Frequency(1, FrequencyUnit.GIGAHERTZ)
array_geometry = geometry.CustomGeometry(x.df.to_numpy().ravel(), y.df.to_numpy().ravel())

arr = Array(antenna_element, array_geometry)

arr.plot.beam(
    DirectionDomain.ANGLE,
    PhaseUnit.DEGREE,
    AmplitudeDomain.AntennaFactor,
    AmplitudeUnit.DECIBEL,
    FigureType.SURFACE,
    Frequency(1, FrequencyUnit.GIGAHERTZ),
)

arr.plot.geometry()
